# Lecke 09 - Metakogníciós tervezési minta


## Beállítás

Ez a jegyzetfüzet bemutatja a metakogníció tervezési mintáját a Microsoft Agent Framework használatával.

**Előfeltételek:**
- Azure OpenAI telepítése környezeti változókkal konfigurálva
- Azure CLI hitelesítve (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mi a metakogníció?

A metakogníció a **gondolkodásról való gondolkodás**. Mesterséges intelligencia ügynökök esetén ez azt jelenti, hogy olyan ügynököket hozunk létre, amelyek képesek:

- **Önreflexiót gyakorolni** a saját kimeneteikre és érvelési folyamatukra
- **Hibákat észlelni** és elegánsan helyreállni ahelyett, hogy csendben kudarcot vallanának
- **Értékelni**, hogy válaszaik teljesek és hasznosak-e
- **Alkalmazkodni** és módosítani a stratégiájukat, amikor az első megközelítés nem működik (pl. visszatérés egy tartalék rendszerhez)

Egy metakognitív ügynök nemcsak válaszol a kérdésekre — folyamatosan figyeli saját teljesítményét és menet közben korrigálja működését.


## Elsődleges és tartalék eszközök

Egy gyakori metakogníciós mintázat a **tartalék stratégia**. Az ügynök először egy elsődleges eszközt próbál; ha az megbukik (pl. 404-es hiba), az ügynök felismeri a hibát és átlátható módon átvált egy tartalék eszközre.

Ez tükrözi a valós rendszereket, ahol az elsődleges szolgáltatások nem érhetők el, és az ügynököknek önmaguknak kell diagnosztizálniuk a problémát, mielőtt alternatív útvonalat választanak.

Az alábbiakban két járatkereső eszközt definiálunk:
- **Elsődleges** — lefedi Párizst, Tokiót és Barcelonát
- **Tartalék** — lefedi Berlint, Sydneyt és New York városát


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Önreflektáló ügynök hibajavítással

A lent látható ügynöknek az a feladata, hogy először az elsődleges repülési rendszert próbálja ki, felismerje a hibákat, és átláthatóan visszatérjen a tartalék rendszerhez. Minden válasz után röviden önértékeli, hogy teljes mértékben megválaszolta-e a felhasználó kérdését.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Önértékelési minta

A metakogníció egy másik aspektusa az **önértékelés**: egy külön ügynök (vagy ugyanaz az ügynök egy második áttekintés során) áttekinti a választ a teljesség, pontosság és hasznosság szempontjából.

Alább létrehozunk egy `ResponseEvaluator` ügynököt, amely három dimenzió mentén értékeli az utazási ügynök válaszait.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Összefoglaló

Ebben a leckében megtanultad, hogyan építs **metakognitív ügynököket** a Microsoft Agent Framework segítségével:

- **Önreflexió**: Olyan ügynökök, amelyek figyelik saját gondolkodásukat és átláthatóan kommunikálják, mi történt.
- **Hibahelyreállítás tartalékokkal**: Egy elsődleges + tartalék eszközminta, ahol az ügynök felismeri a hibákat (pl. 404-es hibákat) és automatikusan megpróbál egy alternatív forrást.
- **Önértékelés**: Egy külön értékelő ügynök, amely pontozza a válaszokat teljesség, pontosság és hasznosság szempontjából.

Ezek a minták robusztusabbá, átláthatóbbá és megbízhatóbbá teszik az ügynököket — kritikus tulajdonságok az éles telepítésekhez.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Felelősségkizárás:
Ezt a dokumentumot az AI-alapú fordító szolgáltatás, a Co-op Translator (https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti, anyanyelvi dokumentum tekintendő hiteles forrásnak. Kritikus fontosságú információk esetén szakmai, emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely a fordítás használatából ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
